# Stage 2c: positions, traffic, overtakes and rivals

This notebook is the last part of stage 2. It takes the lap table and the pit-stop table of `02b_race_situation.ipynb` and adds what depends on the other cars: the race position and the gaps at the end of every lap, whether a lap was driven in traffic, the on-track overtakes, and for every pit stop whether a rival was close. It also puts the cleaning table of the whole stage together.

Principle: *flag, don't delete*. No lap and no stop is removed here.

Input: `data/processed/laps_02b.parquet` and `data/processed/stops_02b.parquet` (from `02b_race_situation.ipynb`), `data/processed/cleaning_02a.csv` and `data/processed/cleaning_02b.csv`; the position, intervals and overtakes files in `data/raw/` (used as checks only). Output: `data/processed/laps.parquet`, `data/processed/pit_stops.parquet`, `data/processed/overtakes.parquet` and `results/cleaning_table.csv`.

In [1]:
import time

import numpy as np
import pandas as pd

from helpers import DATA_PROCESSED, KEY, LAP_KEY, RESULTS, add_step, load_raw

start_time = time.time()

# A rival is the car directly ahead or behind in race position within this many seconds.
RIVAL_GAP = 3.0
# A lap is in traffic when the previous car across the line is closer than this (seconds).
TRAFFIC_GAP = 2.0
# A lap number is a slow field lap when at least this share of the field is an outlier on it.
SLOW_FIELD_SHARE = 0.5
# A pass is "on a slow defender" when the attacker was further back than this before the lap (seconds).
DEFENDER_SLOW_GAP = 3.0
# When comparing with OpenF1's intervals, only a record this close to the lap end is used (seconds).
INTERVAL_WINDOW = 5.0
# Positions closer than this at the line are called a near dead heat in the check (seconds).
DEAD_HEAT_GAP = 0.3
# The three races shown in detail in the checks.
EXAMPLE_RACES = [(2025, "Monza"), (2023, "Jeddah"), (2026, "Monte Carlo")]

# The columns that identify one lap number of one race (all cars).
RACE_LAP = ["session_key", "lap_number"]

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

# The rows this notebook adds to the cleaning table: one row per step.
cleaning_steps = []

## 1. Load the tables of 02b and the raw files used as checks

OpenF1's own position, intervals and overtakes data are loaded only to check our results against them. The interval of a lapped car is a text such as `+1 LAP`; it becomes an empty value.

In [2]:
laps = pd.read_parquet(DATA_PROCESSED / "laps_02b.parquet")
pit_stops = pd.read_parquet(DATA_PROCESSED / "stops_02b.parquet")
laps = laps.sort_values(LAP_KEY).reset_index(drop=True)

position_feed = load_raw("position", columns=["session_key", "driver_number", "date", "position"],
                         number_columns=["position"], date_columns=["date"])
intervals_feed = load_raw("intervals", columns=["session_key", "driver_number", "date", "interval"],
                          date_columns=["date"])
intervals_feed["interval"] = pd.to_numeric(intervals_feed["interval"], errors="coerce")
openf1_overtakes = load_raw(
    "overtakes", columns=["session_key", "overtaking_driver_number", "overtaken_driver_number", "date"],
    date_columns=["date"])

for name, table in [("laps", laps), ("stops", pit_stops), ("OpenF1 position", position_feed),
                    ("OpenF1 intervals", intervals_feed), ("OpenF1 overtakes", openf1_overtakes)]:
    print(f"{name}: {len(table):,} rows, {table['session_key'].nunique()} races")

laps: 101,930 rows, 91 races
stops: 3,418 rows, 91 races
OpenF1 position: 45,248 rows, 91 races
OpenF1 intervals: 2,210,505 rows, 91 races
OpenF1 overtakes: 23,029 rows, 91 races


## 2. Position and gaps at the end of every lap

The lap end time (`lap_end`, from `02a_clean_laps.ipynb`) is the moment a car crosses the line. Among the cars that completed lap *n*, the car that crossed the line first is ahead in the race, so:

- `position` is the rank of the lap end time within the same lap number of the race;
- `gap_ahead` is the difference in seconds to the lap end of the car one position ahead **on the same lap number**, `gap_behind` the same to the car one position behind. For a lapped car the gap to the car ahead can therefore be longer than a lap;
- `car_ahead` and `car_behind` are the driver numbers of those cars, `ahead_is_teammate` and `behind_is_teammate` say whether they drive for the same team.

**Ties.** Some lap end times are shared to the millisecond by two cars (the next lap's start time was delivered with the same value). Their order is taken from `date_start + lap_duration`, the car's own estimate of when the lap ended. `shared_lap_end` marks these laps; their gap to each other is 0.

A lap without a lap end (retirement laps, and lap 1 of 2025 Miami) gets no position and no gaps.

In [3]:
laps["own_end"] = laps["date_start"] + pd.to_timedelta(laps["lap_duration"], unit="s")

crossed = laps[laps["lap_end"].notna()].sort_values(RACE_LAP + ["lap_end", "own_end"]).copy()
by_lap = crossed.groupby(RACE_LAP)
crossed["position"] = by_lap.cumcount() + 1
crossed["car_ahead"] = by_lap["driver_number"].shift(1)
crossed["car_behind"] = by_lap["driver_number"].shift(-1)
crossed["gap_ahead"] = (crossed["lap_end"] - by_lap["lap_end"].shift(1)).dt.total_seconds()
crossed["gap_behind"] = (by_lap["lap_end"].shift(-1) - crossed["lap_end"]).dt.total_seconds()
crossed["ahead_is_teammate"] = by_lap["team"].shift(1) == crossed["team"]
crossed["behind_is_teammate"] = by_lap["team"].shift(-1) == crossed["team"]
crossed["shared_lap_end"] = crossed.duplicated(RACE_LAP + ["lap_end"], keep=False)
# A tie stays undecided when the own estimate is missing or also identical.
undecided = crossed["shared_lap_end"] & (crossed["own_end"].isna()
                                         | crossed.duplicated(RACE_LAP + ["lap_end", "own_end"], keep=False))

print("laps:", len(laps), "| with a lap end, so with a position:", len(crossed),
      "| without:", int(laps["lap_end"].isna().sum()))
print("laps without a position: retirement laps:", int((laps["lap_end"].isna() & laps["incomplete_lap"]).sum()),
      "| others:", int((laps["lap_end"].isna() & ~laps["incomplete_lap"]).sum()))
print(laps[laps["lap_end"].isna() & ~laps["incomplete_lap"]].groupby(["year", "circuit", "lap_number"]).size().to_string())
print("\nlap ends shared with another car:", int(crossed["shared_lap_end"].sum()),
      "| order decided by date_start + lap_duration:", int((crossed["shared_lap_end"] & ~undecided).sum()),
      "| still undecided:", int(undecided.sum()))

laps: 101930 | with a lap end, so with a position: 101669 | without: 261
laps without a position: retirement laps: 241 | others: 20
year  circuit     lap_number
2024  Interlagos  31             1
2025  Miami       1             19

lap ends shared with another car: 120 | order decided by date_start + lap_duration: 102 | still undecided: 18


## 3. Traffic

A lap is in traffic (dirty air) when the car **physically ahead on track** is close. That car is the previous car to cross the line, whatever lap it is on, so a backmarker or a car that is about to be lapped counts too. `track_gap` is the time in seconds since the previous car of the race crossed the line and `car_on_track_ahead` its driver number; `in_traffic` is set when `track_gap` is below 2.0 s.

A car ahead that crosses the line in the pit lane (its in-lap) still counts: it was ahead on track for almost the whole lap.

`clean_lap` is not changed. The pace model of stage 3 uses the laps that are clean **and** not in traffic.

In [4]:
on_track = crossed.sort_values(["session_key", "lap_end", "own_end"])
by_race = on_track.groupby("session_key")
crossed["track_gap"] = by_race["lap_end"].diff().dt.total_seconds()
crossed["car_on_track_ahead"] = by_race["driver_number"].shift(1)
crossed["in_traffic"] = crossed["track_gap"] < TRAFFIC_GAP

position_columns = ["position", "car_ahead", "car_behind", "gap_ahead", "gap_behind", "ahead_is_teammate",
                    "behind_is_teammate", "shared_lap_end", "track_gap", "car_on_track_ahead", "in_traffic"]
laps = laps.merge(crossed[LAP_KEY + position_columns], on=LAP_KEY, how="left", validate="one_to_one")
for column in ["position", "car_ahead", "car_behind", "car_on_track_ahead"]:
    laps[column] = laps[column].astype("Int64")
for column in ["ahead_is_teammate", "behind_is_teammate", "shared_lap_end", "in_traffic"]:
    laps[column] = laps[column].eq(True)

clean = laps[laps["clean_lap"]]
dry_clean = clean[~clean["wet_race"]]
print("gap to the previous car across the line, clean laps (seconds):")
print(clean["track_gap"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2).to_string())
print("\nclean laps:", len(clean), "| in dry races:", len(dry_clean))
for threshold in [1.0, 1.5, TRAFFIC_GAP]:
    print(f"previous car closer than {threshold} s: {int((clean['track_gap'] < threshold).sum())} clean laps "
          f"({(clean['track_gap'] < threshold).mean():.1%}) | dry races: {int((dry_clean['track_gap'] < threshold).sum())} "
          f"({(dry_clean['track_gap'] < threshold).mean():.1%})")
other_lap = dry_clean["in_traffic"] & ((dry_clean["car_on_track_ahead"] != dry_clean["car_ahead"]).fillna(True))
print(f"\nclean dry laps in traffic: {int(dry_clean['in_traffic'].sum())} | of which the car on track ahead is not the car "
      f"one position ahead on the same lap: {int(other_lap.sum())}")
print("clean dry laps left for the pace model (clean and not in traffic):", int((~dry_clean["in_traffic"]).sum()))

add_step(cleaning_steps, "laps without a race position", len(laps), laps["position"].isna().sum(),
         "position and gaps left empty", "no lap end time: retirement laps and lap 1 of 2025 Miami")
add_step(cleaning_steps, "lap end time shared with another car", len(laps), laps["shared_lap_end"].sum(),
         "flagged shared_lap_end", "two cars with the same lap end time; order taken from date_start + lap_duration")
add_step(cleaning_steps, "laps in traffic", len(laps), laps["in_traffic"].sum(), "flagged in_traffic",
         f"the previous car across the line, on any lap, is less than {TRAFFIC_GAP} s ahead")
add_step(cleaning_steps, "of which clean laps", len(laps), (laps["in_traffic"] & laps["clean_lap"]).sum(),
         "flagged in_traffic", "clean laps that stage 3 leaves out of the pace model because of traffic")

gap to the previous car across the line, clean laps (seconds):
count    85950.00
mean         4.77
std          8.43
min          0.00
10%          0.61
25%          1.05
50%          2.29
75%          4.97
90%          9.99
max        106.77

clean laps: 85951 | in dry races: 76926
previous car closer than 1.0 s: 20184 clean laps (23.5%) | dry races: 18114 (23.5%)
previous car closer than 1.5 s: 30501 clean laps (35.5%) | dry races: 27321 (35.5%)
previous car closer than 2.0 s: 38808 clean laps (45.2%) | dry races: 34812 (45.3%)

clean dry laps in traffic: 34812 | of which the car on track ahead is not the car one position ahead on the same lap: 2731
clean dry laps left for the pace model (clean and not in traffic): 42114


## 4. Check against OpenF1's position and intervals data

OpenF1's position data starts about an hour before the race and records only changes, so for each car the last record at or before its lap end is taken (forward-fill). It is a live running order: it also changes between two crossings of the line, for example when a car is passed into the first corner a few seconds after the line.

Every disagreement is put into the first category that fits:

1. `race with laps missing at the source`: 2025 Miami and 2023 Monza;
2. `at or after the chequered flag`: the lap ends when the winner has finished. OpenF1 then shows the classification, with time penalties applied, and odd values for lapped cars;
3. `lap 1`;
4. `red-flag lap`;
5. `shared lap end`: the two cars have the same lap end time;
6. `time_mismatch`: the lap of the car, or of the car directly ahead or behind, is flagged `time_mismatch`;
7. `near dead heat`: the car ahead or behind is less than 0.3 s away at the line;
8. `feed late by up to 2 s` and `feed late by up to 5 s`: OpenF1 shows our position 2 or 5 seconds later;
9. `in-lap or out-lap`: the line is crossed in the pit lane;
10. `other`.

In [5]:
def feed_at_lap_end(lap_table, feed, seconds_later=0, direction="backward"):
    """Return, for each row of lap_table, the feed record of the same car at its lap end (plus some seconds)."""
    moments = lap_table[KEY].copy()
    moments["row"] = lap_table.index
    moments["moment"] = lap_table["lap_end"] + pd.Timedelta(seconds=seconds_later)
    found = pd.merge_asof(moments.sort_values("moment"), feed.sort_values("date"), left_on="moment",
                          right_on="date", by=KEY, direction=direction)
    return found.set_index("row").reindex(lap_table.index)


check = laps[laps["position"].notna()].copy()
check["openf1"] = feed_at_lap_end(check, position_feed)["position"]
check["openf1_2s_later"] = feed_at_lap_end(check, position_feed, seconds_later=2)["position"]
check["openf1_5s_later"] = feed_at_lap_end(check, position_feed, seconds_later=5)["position"]
check["agree"] = check["position"] == check["openf1"]

# The chequered flag falls when the winner ends the last lap of the race.
last_lap = check[check["lap_number"] == check["race_laps"]]
chequered_flag = last_lap.groupby("session_key")["lap_end"].min()
check["after_flag"] = check["lap_end"] >= check["session_key"].map(chequered_flag)

in_order = check.sort_values(RACE_LAP + ["position"])
neighbours = in_order.groupby(RACE_LAP)["time_mismatch"]
mismatch_near = (in_order["time_mismatch"] | neighbours.shift(1, fill_value=False)
                 | neighbours.shift(-1, fill_value=False)).reindex(check.index)
dead_heat = check[["gap_ahead", "gap_behind"]].min(axis=1) < DEAD_HEAT_GAP

categories = [
    ("race with laps missing at the source", check["race_missing_laps"] > 0),
    ("at or after the chequered flag", check["after_flag"]),
    ("lap 1", check["lap_number"] == 1),
    ("red-flag lap", check["track_status"] == "red"),
    ("shared lap end", check["shared_lap_end"]),
    ("time_mismatch", mismatch_near),
    ("near dead heat", dead_heat),
    ("feed late by up to 2 s", check["position"] == check["openf1_2s_later"]),
    ("feed late by up to 5 s", check["position"] == check["openf1_5s_later"]),
    ("in-lap or out-lap", check["is_in_lap"] | check["is_out_lap"]),
]
check["category"] = np.select([condition for _, condition in categories], [name for name, _ in categories],
                              default="other")
disagree = check[~check["agree"]]

print(f"lap ends with a position: {len(check)} | agree with OpenF1: {int(check['agree'].sum())} "
      f"({check['agree'].mean():.2%}) | disagree: {len(disagree)}")
print(f"before the chequered flag: {check.loc[~check['after_flag'], 'agree'].mean():.2%} of "
      f"{int((~check['after_flag']).sum())} | at or after it: {check.loc[check['after_flag'], 'agree'].mean():.2%} of "
      f"{int(check['after_flag'].sum())}")
per_race = check.groupby("session_key")["agree"].mean()
print("races without any disagreement:", int((per_race == 1).sum()), "of", len(per_race),
      f"| lowest agreement in a race: {per_race.min():.2%}")
print("\nthe three example races:")
for year, circuit in EXAMPLE_RACES:
    race = check[(check["year"] == year) & (check["circuit"] == circuit)]
    print(f"  {year} {circuit}: {len(race)} lap ends, {int(race['agree'].sum())} agree ({race['agree'].mean():.2%}), "
          f"{int((~race['agree']).sum())} disagree")
print("\ndisagreements by category:")
print(disagree["category"].value_counts().reindex([name for name, _ in categories] + ["other"]).fillna(0).astype(int).to_string())
shown = ["year", "circuit", "lap_number", "driver", "position", "openf1", "openf1_5s_later", "gap_ahead", "gap_behind",
         "track_status"]
print("\nexamples from the three example races:")
examples = pd.concat([disagree[(disagree["year"] == year) & (disagree["circuit"] == circuit)] for year, circuit in EXAMPLE_RACES])
print(examples.sort_values(["year", "lap_number", "position"])[shown + ["category"]].round(3).to_string(index=False))
print("\nall disagreements in the category 'other':")
print(disagree[disagree["category"] == "other"].sort_values(["year", "circuit", "lap_number", "position"])[shown].round(3).to_string(index=False))

lap ends with a position: 101669 | agree with OpenF1: 101480 (99.81%) | disagree: 189
before the chequered flag: 99.88% of 100080 | at or after it: 95.72% of 1589
races without any disagreement: 41 of 91 | lowest agreement in a race: 98.14%

the three example races:
  2025 Monza: 974 lap ends, 973 agree (99.90%), 1 disagree
  2023 Jeddah: 942 lap ends, 940 agree (99.79%), 2 disagree
  2026 Monte Carlo: 1445 lap ends, 1440 agree (99.65%), 5 disagree

disagreements by category:
category
race with laps missing at the source    14
at or after the chequered flag          68
lap 1                                   15
red-flag lap                             2
shared lap end                          25
time_mismatch                            8
near dead heat                          25
feed late by up to 2 s                  12
feed late by up to 5 s                   6
in-lap or out-lap                        4
other                                   10

examples from the three example race

**The ten disagreements in the category `other`**, looked at one by one in OpenF1's raw position records:

- *2024 Hungaroring, laps 7 (TSU), 28 (STR) and 47 (NOR), and 2025 Mexico City, lap 27 (RUS): OpenF1 is late while a neighbour pits.* The car directly ahead or behind crosses the line in the pit lane (its in-lap). OpenF1 writes our position 3 to 4 s after the car has crossed the line and replaces it within a fraction of a second, when the car that is stopping drops back. Our position at the line never shows up at the moments compared (0, 2 and 5 s later). Our order is the order of crossing the line.
- *2026 Hungaroring, lap 40 (VER, ANT) and lap 41 (NOR): OpenF1 misses a pit stop for a lap.* NOR stops from the lead on lap 39. OpenF1 has no position record for the three cars until the end of lap 41, so it keeps NOR first for a whole lap. The lap end times are clear: NOR's out-lap takes 102.6 s and he crosses the line 10.4 s after VER.
- *2026 Zandvoort, lap 4 (LIN): the restart after the red flag.* LIN crosses the line 51.7 s after the car ahead, last of the field, on the first lap after the red flag. OpenF1 puts him back to 8th (his position before the red flag) at that moment and moves him down to 20th over the next 40 seconds.
- *2025 Sakhir, lap 54 (RUS) and 2023 Zandvoort, lap 28 (HAM): near dead heats just above the 0.3 s limit.* The lap end times put the two cars 0.34 s apart, but their own estimates (`date_start + lap_duration`) are only 0.06 s and 0.01 s apart, which is within the noise of the lap end times (about 0.1 s). The order at the line cannot be decided from the data. RUS also has a timing fault in that race (13 of his laps are flagged `time_mismatch`).

None is left unexplained. In the first three groups our order is the right one at the line; in the last group either order is possible.

**Gaps.** OpenF1's `interval` is its own gap to the car ahead. For each lap end the interval record nearest in time is taken (it is usually written about 0.2 s before the car crosses the line). The comparison uses the laps where both sources agree on the position, the car is not the leader, the interval is a number and its record is within 5 s of the lap end.

In [6]:
nearest = feed_at_lap_end(check, intervals_feed, direction="nearest")
check["openf1_interval"] = nearest["interval"]
check["interval_distance"] = (nearest["date"] - check["lap_end"]).dt.total_seconds()
comparable = check[check["agree"] & (check["position"] > 1) & check["openf1_interval"].notna()
                   & (check["interval_distance"].abs() <= INTERVAL_WINDOW)].copy()
comparable["difference"] = (comparable["gap_ahead"] - comparable["openf1_interval"]).abs()
print("lap ends that are not the leader's:", int((check["position"] > 1).sum()), "| comparable:", len(comparable))
print(f"seconds from the lap end to the interval record, median: {comparable['interval_distance'].median():.2f}")


def gap_summary(table):
    """Return how far our gap is from OpenF1's interval over the rows of table (seconds)."""
    difference = table["difference"]
    return {"laps": len(table), "median": difference.median(), "p95": difference.quantile(0.95),
            "p99": difference.quantile(0.99), "largest": difference.max(),
            "within_0.2s": (difference <= 0.2).mean(), "within_0.5s": (difference <= 0.5).mean()}


normal = ((comparable["track_status"] == "green") & ~comparable["is_in_lap"] & ~comparable["is_out_lap"]
          & (comparable["lap_number"] > 1) & ~comparable["after_flag"])
groups = {f"{year} {circuit}": comparable[(comparable["year"] == year) & (comparable["circuit"] == circuit)]
          for year, circuit in EXAMPLE_RACES}
groups["all races"] = comparable
groups["green, no in/out lap, not lap 1, before the flag"] = comparable[normal]
groups["  of which gap below 3 s"] = comparable[normal & (comparable["gap_ahead"] < RIVAL_GAP)]
groups["in-laps and out-laps"] = comparable[comparable["is_in_lap"] | comparable["is_out_lap"]]
groups["at or after the chequered flag"] = comparable[comparable["after_flag"]]
print("\nabsolute difference between our gap and OpenF1's interval (seconds):")
print(pd.DataFrame({name: gap_summary(table) for name, table in groups.items()}).T.round(3).to_string())

large = comparable[normal & (comparable["difference"] > 1)]
print(f"\nnormal laps differing by more than 1 s: {len(large)} | of which time_mismatch: {int(large['time_mismatch'].sum())}")
close_ours = comparable.loc[normal, "gap_ahead"] <= RIVAL_GAP
close_openf1 = comparable.loc[normal, "openf1_interval"] <= RIVAL_GAP
print(f"car ahead within {RIVAL_GAP} s on normal laps: both sources {int((close_ours & close_openf1).sum())} | "
      f"ours only {int((close_ours & ~close_openf1).sum())} | OpenF1 only {int((~close_ours & close_openf1).sum())}")
print("\nlargest differences in the three example races:")
examples = pd.concat([table.nlargest(4, "difference") for name, table in list(groups.items())[:3]])
print(examples[["year", "circuit", "lap_number", "driver", "position", "gap_ahead", "openf1_interval", "is_in_lap",
                "track_status", "after_flag"]].round(3).to_string(index=False))

lap ends that are not the leader's: 96209 | comparable: 91322
seconds from the lap end to the interval record, median: -0.17

absolute difference between our gap and OpenF1's interval (seconds):
                                                     laps  median    p95    p99   largest  within_0.2s  within_0.5s
2025 Monza                                          853.0   0.054  0.156  0.222    10.001        0.984        0.993
2023 Jeddah                                         852.0   0.026  0.120  0.353     0.506        0.969        0.999
2026 Monte Carlo                                   1245.0   0.049  0.142  0.257    10.086        0.985        0.996
all races                                         91322.0   0.045  0.160  0.311  1079.049        0.972        0.996
green, no in/out lap, not lap 1, before the flag  80516.0   0.044  0.154  0.283    11.783        0.976        0.999
  of which gap below 3 s                          46826.0   0.045  0.156  0.282     6.806        0.975       

The largest differences have known causes: on the last lap OpenF1 adds time penalties to the interval (differences of exactly 5 or 10 s), and on in-laps OpenF1's interval is taken a few seconds before the car reaches the line in the pit lane.

## 5. The restart period after a red flag

When a race is resumed after a red flag, the cars drive one or more laps behind the safety car before the standing or rolling start. Race control has no message for the end of these laps, so they are `green`. `after_red_flag` (from 02b) marks the first two green laps of a car after a red flag, which is not always enough: at 2026 Monte Carlo and 2026 Monza the real restart comes one lap later.

A lap number is a **slow field lap** when at least half of the cars are flagged `outlier` on it (the outlier rule of 02b: slower than 1.07 times the driver's stint median). `red_flag_restart` marks, for every car, the laps from the first green lap after a red flag **up to and including the first lap after the last slow field lap**: it starts with the two `after_red_flag` laps and is extended lap by lap while the lap is a slow field lap or directly follows one. Slow field laps elsewhere in a race (rain) are not touched.

The flag is saved on the lap table so that stage 4 can use the same window.

In [7]:
laps["is_outlier"] = laps["not_clean_reason"] == "outlier"
laps["slow_field_lap"] = laps.groupby(RACE_LAP)["is_outlier"].transform("mean") >= SLOW_FIELD_SHARE
slow_laps = laps.loc[laps["slow_field_lap"], RACE_LAP].drop_duplicates()
after_slow = slow_laps.assign(lap_number=slow_laps["lap_number"] + 1, after_slow_field_lap=True)
laps = laps.drop(columns="after_slow_field_lap", errors="ignore").merge(after_slow, on=RACE_LAP, how="left")
laps["after_slow_field_lap"] = laps["after_slow_field_lap"].eq(True)

by_car = laps.groupby(KEY)
follows_directly = by_car["lap_number"].shift(1) == laps["lap_number"] - 1
may_extend = laps["slow_field_lap"] | laps["after_slow_field_lap"]
laps["red_flag_restart"] = laps["after_red_flag"]
# Extend the window one lap at a time until nothing changes.
while True:
    previous_in_window = laps.groupby(KEY)["red_flag_restart"].shift(1, fill_value=False) & follows_directly
    extended = laps["red_flag_restart"] | (previous_in_window & may_extend)
    if (extended == laps["red_flag_restart"]).all():
        break
    laps["red_flag_restart"] = extended

print("slow field laps (lap numbers of a race):", len(slow_laps))
print("laps in the two-lap rule (after_red_flag):", int(laps["after_red_flag"].sum()),
      "| laps in the restart period (red_flag_restart):", int(laps["red_flag_restart"].sum()),
      "| added:", int((laps["red_flag_restart"] & ~laps["after_red_flag"]).sum()))
window = laps[laps["red_flag_restart"]].groupby(["year", "circuit", "wet_race", "lap_number"]).agg(
    cars=("driver_number", "size"), two_lap_rule=("after_red_flag", "sum"), slow_field_lap=("slow_field_lap", "first"),
    outliers=("is_outlier", "sum"), median_lap_time=("lap_duration", "median"))
print("\nthe restart period by race and lap number (cars = cars with the flag on that lap):")
print(window.round(1).to_string())

stop_laps = laps[LAP_KEY + ["after_red_flag", "red_flag_restart"]].rename(columns={"lap_number": "in_lap"})
stops_in_window = pit_stops.merge(stop_laps, on=KEY + ["in_lap"], how="left")
newly_inside = stops_in_window["red_flag_restart"].eq(True) & ~stops_in_window["after_red_flag"].eq(True)
print("\nstops with the in-lap in the restart period but outside the two-lap rule:", int(newly_inside.sum()),
      "| of which strategy stops:", int((newly_inside & stops_in_window["strategy_stop"]).sum()))
print(stops_in_window.loc[newly_inside, ["year", "circuit", "driver", "in_lap", "compound_before", "compound_after",
                                         "strategy_stop", "not_strategy_reason"]].to_string(index=False))
add_step(cleaning_steps, "laps in the restart period after a red flag", len(laps), laps["red_flag_restart"].sum(),
         "flagged red_flag_restart",
         "from the first green lap after a red flag up to the first lap after the last lap on which at least "
         "half the field is an outlier (laps behind the safety car and the restart)")

slow field laps (lap numbers of a race): 20
laps in the two-lap rule (after_red_flag): 423 | laps in the restart period (red_flag_restart): 554 | added: 131

the restart period by race and lap number (cars = cars with the flag on that lap):
                                      cars  two_lap_rule  slow_field_lap  outliers  median_lap_time
year circuit     wet_race lap_number                                                               
2023 Interlagos  False    3              2             2           False         0            106.5
                          4             17            17           False         0             82.3
                          5             15            15           False         0             77.1
     Melbourne   False    9             17            17           False         0            194.7
                          10            18            18            True        16             91.3
                          11            18             1   

`strategy_stop` is **not** changed here: the stops listed above are only reported.

## 6. On-track overtakes

An on-track overtake is a **swap with the car directly in front** between two consecutive lap ends: at the end of lap *n* − 1 the defender is one position ahead of the attacker, and at the end of lap *n* the attacker is ahead of the defender. Both cars must have a position at the end of both laps, so a car that retires is never overtaken, and passes during lap 1 cannot be seen (there is no lap end before it).

A swap is not an on-track overtake when one of the following holds for the attacker or the defender; the first one that holds is the reason:

1. `pit_stop`: lap *n* is an in-lap or an out-lap;
2. `not_green`: lap *n* is under a safety car, a VSC or a red flag;
3. `restart`: lap *n* is a restart lap;
4. `red_flag_restart`: lap *n* is in the restart period after a red flag (section 5);
5. `time_mismatch`: lap *n* or lap *n* − 1 is flagged `time_mismatch`, so one of the lap end times that decide the swap is unreliable.

Cars on different laps need no rule: the two cars are compared on the same lap number, and an attacker that ends lap *n* ahead of the defender cannot have been a full lap behind at the end of lap *n* − 1.

Kept, with a flag so that stage 4 can decide:
- `teammates`: both cars drive for the same team;
- `defender_slow`: the defender's lap is an outlier, or the defender retires on the next lap, or the attacker was more than 3 s behind before the lap. These are position changes without a real fight;
- `wet_race` and `race_missing_laps`, the race flags.

A swap is the net change over one lap: a pass that is taken back before the line is not seen.

In [8]:
by_car = laps.groupby(KEY)
follows_directly = by_car["lap_number"].shift(1) == laps["lap_number"] - 1
is_followed = by_car["lap_number"].shift(-1) == laps["lap_number"] + 1
laps["timing_doubt"] = laps["time_mismatch"] | (by_car["time_mismatch"].shift(1, fill_value=False) & follows_directly)
laps["retires_next_lap"] = by_car["incomplete_lap"].shift(-1, fill_value=False) & is_followed

placed = laps[laps["position"].notna()]
lap_facts = ["position", "driver", "team", "compound", "tyre_age", "lap_end", "is_in_lap", "is_out_lap", "track_status",
             "restart_lap", "red_flag_restart", "timing_doubt", "not_clean_reason", "retires_next_lap"]


def car_on_lap(role):
    """Return the lap facts of every car, with the columns named after its role in a swap."""
    table = placed[LAP_KEY + lap_facts].rename(columns={"driver_number": role})
    return table.rename(columns={name: f"{role}_{name}" for name in lap_facts})


# One row per car and lap: the car directly in front at the end of the lap before.
pairs = placed.loc[placed["car_ahead"].notna(), LAP_KEY + ["car_ahead", "gap_ahead"]].rename(
    columns={"driver_number": "attacker", "car_ahead": "defender", "gap_ahead": "gap_before"})
pairs["defender"] = pairs["defender"].astype(int)
pairs["lap_number"] = pairs["lap_number"] + 1
pairs = pairs.merge(car_on_lap("attacker"), on=RACE_LAP + ["attacker"], validate="one_to_one")
pairs = pairs.merge(car_on_lap("defender"), on=RACE_LAP + ["defender"], validate="one_to_one")
swaps = pairs[pairs["attacker_position"] < pairs["defender_position"]].copy()
race_facts = laps.groupby("session_key")[["year", "circuit", "wet_race", "race_missing_laps"]].first().reset_index()
swaps = swaps.merge(race_facts, on="session_key", validate="many_to_one")


def either(column):
    """Return True where the attacker's or the defender's lap has the flag."""
    return swaps[f"attacker_{column}"] | swaps[f"defender_{column}"]


swap_conditions = [
    ("pit_stop", either("is_in_lap") | either("is_out_lap")),
    ("not_green", (swaps["attacker_track_status"] != "green") | (swaps["defender_track_status"] != "green")),
    ("restart", either("restart_lap")),
    ("red_flag_restart", either("red_flag_restart")),
    ("time_mismatch", either("timing_doubt")),
]
swaps["not_overtake_reason"] = np.select([condition for _, condition in swap_conditions],
                                         [name for name, _ in swap_conditions], default="")
swap_reasons = pd.DataFrame({
    "swaps_with_this_condition": [int(condition.sum()) for _, condition in swap_conditions],
    "first_reason_for": [int((swaps["not_overtake_reason"] == name).sum()) for name, _ in swap_conditions],
}, index=[name for name, _ in swap_conditions])

overtakes = swaps[swaps["not_overtake_reason"] == ""].copy()
overtakes["gap_after"] = (overtakes["defender_lap_end"] - overtakes["attacker_lap_end"]).dt.total_seconds()
overtakes["teammates"] = overtakes["attacker_team"] == overtakes["defender_team"]
defender_outlier = overtakes["defender_not_clean_reason"] == "outlier"
far_behind = overtakes["gap_before"] > DEFENDER_SLOW_GAP
overtakes["defender_slow"] = defender_outlier | overtakes["defender_retires_next_lap"] | far_behind

print("pairs of cars one position apart with a position on two consecutive laps:", len(pairs))
print("swaps with the car directly in front:", len(swaps), "| on-track overtakes:", len(overtakes))
print(swap_reasons.to_string())
print("\non-track overtakes: between teammates:", int(overtakes["teammates"].sum()),
      "| on a slow defender:", int(overtakes["defender_slow"].sum()),
      f"(defender's lap an outlier: {int(defender_outlier.sum())}, defender retires on the next lap: "
      f"{int(overtakes['defender_retires_next_lap'].sum())}, attacker more than {DEFENDER_SLOW_GAP} s behind before: "
      f"{int(far_behind.sum())})")
print("in wet races:", int(overtakes["wet_race"].sum()), "| in races with laps missing at the source:",
      int((overtakes["race_missing_laps"] > 0).sum()), "| on lap 2:", int((overtakes["lap_number"] == 2).sum()))
print("\ngap between the two cars before and after the lap (seconds):")
print(overtakes[["gap_before", "gap_after"]].describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())

add_step(cleaning_steps, "position swaps that are not on-track overtakes (counts are swaps, not laps)", len(swaps),
         (swaps["not_overtake_reason"] != "").sum(), "left out of the overtakes table",
         "swap with the car directly in front on an in-lap or out-lap, under a safety car, VSC or red flag, on a "
         "restart lap, in the restart period after a red flag, or with a time_mismatch; first reason counted")
add_step(cleaning_steps, "on-track overtakes on a slow defender (counts are overtakes)", len(overtakes),
         overtakes["defender_slow"].sum(), "flagged defender_slow",
         f"the defender's lap is an outlier, the defender retires on the next lap, or the attacker was more than "
         f"{DEFENDER_SLOW_GAP} s behind before the lap")
add_step(cleaning_steps, "on-track overtakes between teammates (counts are overtakes)", len(overtakes),
         overtakes["teammates"].sum(), "flagged teammates", "attacker and defender drive for the same team")

overtake_columns = {
    "year": "year", "session_key": "session_key", "circuit": "circuit", "lap_number": "lap_number",
    "attacker": "attacker", "attacker_driver": "attacker_driver", "defender": "defender",
    "defender_driver": "defender_driver", "attacker_position": "position_after",
    "attacker_compound": "attacker_compound", "attacker_tyre_age": "attacker_tyre_age",
    "defender_compound": "defender_compound", "defender_tyre_age": "defender_tyre_age",
    "gap_before": "gap_before", "gap_after": "gap_after", "teammates": "teammates",
    "defender_slow": "defender_slow", "wet_race": "wet_race", "race_missing_laps": "race_missing_laps"}
overtakes = overtakes[list(overtake_columns)].rename(columns=overtake_columns)
overtakes = overtakes.sort_values(["year", "session_key", "lap_number", "position_after"]).reset_index(drop=True)

pairs of cars one position apart with a position on two consecutive laps: 94355
swaps with the car directly in front: 6469 | on-track overtakes: 2919
                  swaps_with_this_condition  first_reason_for
pit_stop                               3294              3294
not_green                               485                54
restart                                 131               123
red_flag_restart                         77                42
time_mismatch                            73                37

on-track overtakes: between teammates: 251 | on a slow defender: 114 (defender's lap an outlier: 91, defender retires on the next lap: 47, attacker more than 3.0 s behind before: 30)
in wet races: 354 | in races with laps missing at the source: 33 | on lap 2: 106

gap between the two cars before and after the lap (seconds):
       gap_before  gap_after
count     2919.00    2919.00
mean         0.59       1.82
std          0.70       3.81
min          0.00       0.00
50%   

**How many per race.** The table has one row per race, with races that have no overtake counted as zero.

In [9]:
race_list = laps.groupby("session_key").agg(year=("year", "first"), circuit=("circuit", "first"),
                                            wet_race=("wet_race", "first"), laps=("race_laps", "first"))
race_list["overtakes"] = overtakes.groupby("session_key").size().reindex(race_list.index).fillna(0).astype(int)
race_list["on_slow_defender"] = overtakes.groupby("session_key")["defender_slow"].sum().reindex(race_list.index).fillna(0).astype(int)
ordered = race_list.sort_values("overtakes")
print("on-track overtakes per race:")
print(race_list["overtakes"].describe().round(1).to_string())
print("\nthe five races with the fewest:")
print(ordered.head(5).to_string(index=False))
print("\nthe five races with the most:")
print(ordered.tail(5).iloc[::-1].to_string(index=False))
print("\nper season:")
print(race_list.groupby("year").agg(races=("overtakes", "size"), overtakes=("overtakes", "sum"),
                                    per_race=("overtakes", "mean")).round(1).to_string())
by_circuit = race_list.groupby("circuit").agg(races=("overtakes", "size"), per_race=("overtakes", "mean"),
                                              fewest=("overtakes", "min"), most=("overtakes", "max"))
print("\nper race by circuit, all seasons (ordered by the mean per race):")
print(by_circuit.sort_values("per_race").round(1).to_string())

on-track overtakes per race:
count    91.0
mean     32.1
std      15.3
min       3.0
25%      23.0
50%      31.0
75%      39.5
max      95.0

the five races with the fewest:
 year     circuit  wet_race  laps  overtakes  on_slow_defender
 2026 Monte Carlo     False    78          3                 0
 2025 Monte Carlo     False    78          4                 0
 2024 Monte Carlo     False    78          5                 0
 2025      Lusail     False    57          6                 1
 2023 Monte Carlo      True    78         10                 4

the five races with the most:
 year            circuit  wet_race  laps  overtakes  on_slow_defender
 2026          Las Vegas     False    50         95                 2
 2023          Las Vegas     False    50         64                 0
 2026              Monza     False    53         61                 1
 2023 Yas Marina Circuit     False    58         59                 0
 2026         Interlagos      True    71         59                

**Cross-check with OpenF1's overtakes.** OpenF1's overtakes data counts every change in the timing order, including the start and pit stops, so it is much larger and is not a source. But a real on-track pass should appear in it. Each OpenF1 record is put on the lap the overtaking car was driving at the time, and one of our overtakes is found when OpenF1 has the same attacker and defender on the same lap, the lap before or the lap after.

In [10]:
records = openf1_overtakes.rename(columns={"overtaking_driver_number": "attacker",
                                           "overtaken_driver_number": "defender"})
attacker_laps = placed[LAP_KEY + ["lap_end"]].rename(columns={"driver_number": "attacker"})
records = pd.merge_asof(records.sort_values("date"), attacker_laps.sort_values("lap_end"), left_on="date",
                        right_on="lap_end", by=["session_key", "attacker"], direction="forward")
print("OpenF1 overtake records:", len(records), "| on lap 1:", int((records["lap_number"] == 1).sum()),
      "| after the attacker's last lap end:", int(records["lap_number"].isna().sum()),
      "| from lap 2 on:", int((records["lap_number"] > 1).sum()))

pair_key = ["session_key", "attacker", "defender", "lap_number"]
known = records.dropna(subset=["lap_number"])[pair_key].astype(int).drop_duplicates()
found = pd.Series(False, index=overtakes.index)
for shift in [-1, 0, 1]:
    shifted = known.assign(lap_number=known["lap_number"] + shift, in_openf1=True)
    found |= overtakes[pair_key].merge(shifted, on=pair_key, how="left")["in_openf1"].eq(True).to_numpy()
print(f"our on-track overtakes: {len(overtakes)} | also in OpenF1's overtakes: {int(found.sum())} ({found.mean():.1%})")
print("ours that are found in OpenF1, per season:")
print(found.groupby(overtakes["year"]).agg(["size", "sum", "mean"]).rename(
    columns={"size": "overtakes", "sum": "in_openf1", "mean": "share"}).round(3).to_string())
for year, circuit in EXAMPLE_RACES:
    race_key = race_list[(race_list["year"] == year) & (race_list["circuit"] == circuit)].index[0]
    ours = overtakes["session_key"] == race_key
    print(f"{year} {circuit}: OpenF1 records {int((records['session_key'] == race_key).sum())} | swaps with the car in front "
          f"{int((swaps['session_key'] == race_key).sum())} | on-track overtakes {int(ours.sum())}, of which in OpenF1 "
          f"{int(found[ours].sum())}")
print("\nexamples of our overtakes that OpenF1 does not have:")
print(overtakes[~found].head(8)[["year", "circuit", "lap_number", "attacker_driver", "defender_driver", "gap_before",
                                 "gap_after", "defender_slow"]].round(2).to_string(index=False))

OpenF1 overtake records: 23029 | on lap 1: 4387 | after the attacker's last lap end: 252 | from lap 2 on: 18390


our on-track overtakes: 2919 | also in OpenF1's overtakes: 2886 (98.9%)
ours that are found in OpenF1, per season:
      overtakes  in_openf1  share
year                             
2023        742        740  0.997
2024        711        698  0.982
2025        663        649  0.979
2026        803        799  0.995
2025 Monza: OpenF1 records 202 | swaps with the car in front 58 | on-track overtakes 26, of which in OpenF1 25
2023 Jeddah: OpenF1 records 168 | swaps with the car in front 61 | on-track overtakes 31, of which in OpenF1 31
2026 Monte Carlo: OpenF1 records 132 | swaps with the car in front 30 | on-track overtakes 3, of which in OpenF1 3

examples of our overtakes that OpenF1 does not have:
 year            circuit  lap_number attacker_driver defender_driver  gap_before  gap_after  defender_slow
 2023          Spielberg          19             STR             BOT        0.44       1.25          False
 2023 Yas Marina Circuit          22             RUS             SAI       

## 7. Rivals at every pit stop

For every stop, the situation at the **end of the lap before the in-lap**: the position, the cars directly ahead and behind in race position and the gaps to them (seconds).

A rival is the car directly ahead or behind within 3 s that is not a teammate. A lapped car is never a rival: the gaps are measured on the same lap number, so a car within 3 s is on the same lap. `rival_ahead` and `rival_behind` say on which side a rival is, `rival_close` that there is at least one, and `rival_side` is `ahead`, `behind`, `both` or `none`. A stop whose car has no position on the lap before (a stop at the end of lap 1, or a missing lap) gets `rival_side` = `unknown` and `rival_close` = False.

In [11]:
before_stop = laps[LAP_KEY + ["position", "car_ahead", "car_behind", "gap_ahead", "gap_behind", "ahead_is_teammate",
                              "behind_is_teammate", "time_mismatch"]].rename(
    columns={"lap_number": "lap_before", "position": "position_before", "time_mismatch": "lap_before_time_mismatch"})
new_stop_columns = [column for column in before_stop.columns if column not in KEY] + [
    "rival_ahead", "rival_behind", "rival_close", "rival_side"]
pit_stops = pit_stops.drop(columns=new_stop_columns, errors="ignore")
pit_stops["lap_before"] = pit_stops["in_lap"] - 1
pit_stops = pit_stops.merge(before_stop, on=KEY + ["lap_before"], how="left", validate="many_to_one")
for column in ["ahead_is_teammate", "behind_is_teammate", "lap_before_time_mismatch"]:
    pit_stops[column] = pit_stops[column].eq(True)

pit_stops["rival_ahead"] = (pit_stops["gap_ahead"] <= RIVAL_GAP) & ~pit_stops["ahead_is_teammate"]
pit_stops["rival_behind"] = (pit_stops["gap_behind"] <= RIVAL_GAP) & ~pit_stops["behind_is_teammate"]
pit_stops["rival_close"] = pit_stops["rival_ahead"] | pit_stops["rival_behind"]
pit_stops["rival_side"] = np.select(
    [pit_stops["position_before"].isna(), pit_stops["rival_ahead"] & pit_stops["rival_behind"],
     pit_stops["rival_ahead"], pit_stops["rival_behind"]],
    ["unknown", "both", "ahead", "behind"], default="none")

strategy = pit_stops[pit_stops["strategy_stop"]]
sides = ["ahead", "behind", "both", "none", "unknown"]
print("stops:", len(pit_stops), "| strategy stops:", len(strategy))
print("\nrival within", RIVAL_GAP, "s at the end of the lap before the in-lap:")
side_table = pd.DataFrame({"all_stops": pit_stops["rival_side"].value_counts(),
                           "strategy_stops": strategy["rival_side"].value_counts()}).reindex(sides).fillna(0).astype(int)
side_table["share_of_strategy_stops"] = (side_table["strategy_stops"] / len(strategy)).round(3)
print(side_table.to_string())
print(f"\nstrategy stops with a rival close: {int(strategy['rival_close'].sum())} ({strategy['rival_close'].mean():.1%})")
print("strategy stops per season by rival_side:")
print(strategy.groupby(["year", "rival_side"]).size().unstack(fill_value=0).reindex(columns=sides, fill_value=0).to_string())
teammate_ahead = (strategy["gap_ahead"] <= RIVAL_GAP) & strategy["ahead_is_teammate"]
teammate_behind = (strategy["gap_behind"] <= RIVAL_GAP) & strategy["behind_is_teammate"]
print(f"\nstrategy stops with a teammate within {RIVAL_GAP} s: ahead {int(teammate_ahead.sum())}, behind "
      f"{int(teammate_behind.sum())} | of which no rival on either side: "
      f"{int(((teammate_ahead | teammate_behind) & ~strategy['rival_close']).sum())}")
print("strategy stops whose lap before is flagged time_mismatch (gaps less reliable):",
      int(strategy["lap_before_time_mismatch"].sum()))
print("strategy stops without a position on the lap before:", int(strategy["position_before"].isna().sum()))
print(strategy.loc[strategy["position_before"].isna(), ["year", "circuit", "driver", "in_lap"]].to_string(index=False))

add_step(cleaning_steps, "pit stops with a rival close (counts are stops, not laps)", len(pit_stops),
         pit_stops["rival_close"].sum(), "flagged rival_close",
         f"the car directly ahead or behind at the end of the lap before the in-lap is within {RIVAL_GAP} s and "
         f"not a teammate; side in rival_side")
add_step(cleaning_steps, "pit stops without a position on the lap before (counts are stops)", len(pit_stops),
         pit_stops["position_before"].isna().sum(), "flagged rival_side = unknown",
         "stop at the end of lap 1, or the lap before the in-lap has no lap end time")

stops: 3418 | strategy stops: 1873

rival within 3.0 s at the end of the lap before the in-lap:
            all_stops  strategy_stops  share_of_strategy_stops
rival_side                                                    
ahead             534             283                    0.151
behind            757             487                    0.260
both              858             488                    0.261
none             1150             615                    0.328
unknown           119               0                    0.000

strategy stops with a rival close: 1258 (67.2%)
strategy stops per season by rival_side:
rival_side  ahead  behind  both  none  unknown
year                                          
2023           72     128   140   175        0
2024           81     149   150   163        0
2025           71     124   156   129        0
2026           59      86    42   148        0

strategy stops with a teammate within 3.0 s: ahead 61, behind 120 | of which no rival on e

## 8. Checks

In [12]:
# 1. One row per driver per lap, and nothing was added or lost.
assert not laps.duplicated(LAP_KEY).any()
assert len(laps) == len(pd.read_parquet(DATA_PROCESSED / "laps_02b.parquet", columns=["session_key"]))
assert len(pit_stops) == len(pd.read_parquet(DATA_PROCESSED / "stops_02b.parquet", columns=["session_key"]))
print("1. laps:", len(laps), "| stops:", len(pit_stops), "| same numbers as in the files of 02b: True")

# 2. On every lap number of a race the positions are 1, 2, 3, ... without gaps or repeats.
placed = laps[laps["position"].notna()]
per_lap = placed.groupby(RACE_LAP)["position"].agg(["size", "max", "nunique"])
assert (per_lap["size"] == per_lap["max"]).all() and (per_lap["size"] == per_lap["nunique"]).all()
assert (placed["gap_ahead"].dropna() >= 0).all() and (placed["track_gap"].dropna() >= 0).all()
assert (placed["gap_ahead"].isna() == (placed["position"] == 1)).all()
print("2. positions run from 1 without gaps on every lap of every race; no negative gap; only the leader has no gap ahead")

# 3. A car is never its own neighbour, and teammates are in the same team.
assert (placed["car_ahead"] != placed["driver_number"]).all() and (placed["car_behind"] != placed["driver_number"]).all()
print("3. laps with a teammate directly ahead:", int(laps["ahead_is_teammate"].sum()),
      "| directly behind:", int(laps["behind_is_teammate"].sum()))
assert laps["ahead_is_teammate"].sum() == laps["behind_is_teammate"].sum()

# 4. Agreement with OpenF1's position data (section 4).
print(f"4. positions agree with OpenF1 at {check['agree'].mean():.2%} of lap ends "
      f"({check.loc[~check['after_flag'], 'agree'].mean():.2%} before the chequered flag); "
      f"disagreements in the category 'other': {int((disagree['category'] == 'other').sum())}")

# 5. Overtakes: attacker and defender differ, both laps are green racing laps, and no pair appears twice on a lap.
assert (overtakes["attacker"] != overtakes["defender"]).all()
assert not overtakes.duplicated(["session_key", "lap_number", "attacker"]).any()
assert (overtakes["lap_number"] >= 2).all() and (overtakes["gap_before"] >= 0).all() and (overtakes["gap_after"] >= 0).all()
print("5. on-track overtakes:", len(overtakes), "| races with at least one:", overtakes["session_key"].nunique(),
      "of", laps["session_key"].nunique())

# 6. Rival flags agree with each other.
assert (pit_stops["rival_close"] == pit_stops["rival_side"].isin(["ahead", "behind", "both"])).all()
assert not (pit_stops["rival_ahead"] & pit_stops["ahead_is_teammate"]).any()
assert not (pit_stops["rival_behind"] & pit_stops["behind_is_teammate"]).any()
print("6. stops with a rival close:", int(pit_stops["rival_close"].sum()), "| strategy stops with a rival close:",
      int((pit_stops["strategy_stop"] & pit_stops["rival_close"]).sum()))

1. laps: 101930 | stops: 3418 | same numbers as in the files of 02b: True


2. positions run from 1 without gaps on every lap of every race; no negative gap; only the leader has no gap ahead
3. laps with a teammate directly ahead: 11109 | directly behind: 11109
4. positions agree with OpenF1 at 99.81% of lap ends (99.88% before the chequered flag); disagreements in the category 'other': 10
5. on-track overtakes: 2919 | races with at least one: 91 of 91
6. stops with a rival close: 2149 | strategy stops with a rival close: 1258


## 9. Save

The three final tables of stage 2, and the cleaning table of the whole stage: the rows of `02a_clean_laps.ipynb`, of `02b_race_situation.ipynb` and of this notebook, in that order.

In [13]:
helper_columns = ["own_end", "is_outlier", "slow_field_lap", "after_slow_field_lap", "timing_doubt", "retires_next_lap"]
laps = laps.drop(columns=helper_columns)
laps = laps.sort_values(["year", "session_key", "driver_number", "lap_number"]).reset_index(drop=True)
pit_stops = pit_stops.drop(columns=["lap_before", "lap_before_time_mismatch"])

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
laps.to_parquet(DATA_PROCESSED / "laps.parquet", index=False)
pit_stops.to_parquet(DATA_PROCESSED / "pit_stops.parquet", index=False)
overtakes.to_parquet(DATA_PROCESSED / "overtakes.parquet", index=False)

steps_02a = pd.read_csv(DATA_PROCESSED / "cleaning_02a.csv")
steps_02b = pd.read_csv(DATA_PROCESSED / "cleaning_02b.csv")
own_steps = pd.DataFrame(cleaning_steps)
cleaning_table = pd.concat([steps_02a, steps_02b, own_steps], ignore_index=True)
cleaning_table.to_csv(RESULTS / "cleaning_table.csv", index=False)
print("saved laps.parquet:", laps.shape, "| pit_stops.parquet:", pit_stops.shape, "| overtakes.parquet:", overtakes.shape)
print(f"saved results/cleaning_table.csv: {len(cleaning_table)} rows ({len(steps_02a)} from 02a, {len(steps_02b)} from 02b, "
      f"{len(own_steps)} from this notebook)")

saved laps.parquet: (101930, 55) | pit_stops.parquet: (3418, 48) | overtakes.parquet: (2919, 19)
saved results/cleaning_table.csv: 37 rows (19 from 02a, 8 from 02b, 10 from this notebook)


## 10. Report

In [14]:
pd.set_option("display.max_colwidth", 140)
print("CLEANING TABLE OF STAGE 2 (results/cleaning_table.csv)")
print(cleaning_table.to_string(index=False))

print("\nPOSITIONS")
print(f"lap ends with a position: {len(check)} | agree with OpenF1's position data: {int(check['agree'].sum())} "
      f"({check['agree'].mean():.2%}) | before the chequered flag: {check.loc[~check['after_flag'], 'agree'].mean():.2%}")
print(disagree["category"].value_counts().to_string())
print(f"gap to the car ahead against OpenF1's interval, normal laps: median difference "
      f"{comparable.loc[normal, 'difference'].median():.3f} s, within 0.5 s on "
      f"{(comparable.loc[normal, 'difference'] <= 0.5).mean():.1%} of {int(normal.sum())} laps")

print("\nTRAFFIC")
clean = laps[laps["clean_lap"]]
dry_clean = clean[~clean["wet_race"]]
print(f"clean laps in traffic: {int(clean['in_traffic'].sum())} of {len(clean)} ({clean['in_traffic'].mean():.1%}) | "
      f"dry races: {int(dry_clean['in_traffic'].sum())} of {len(dry_clean)} ({dry_clean['in_traffic'].mean():.1%})")
print("clean laps in dry races, per season:")
print(dry_clean.groupby("year")["in_traffic"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "clean_laps", "sum": "in_traffic", "mean": "share"}).round(3).to_string())

print("\nOVERTAKES")
print(f"on-track overtakes: {len(overtakes)} | per race: median {race_list['overtakes'].median():.0f}, "
      f"from {race_list['overtakes'].min()} to {race_list['overtakes'].max()} | also in OpenF1's overtakes: {found.mean():.1%}")
print("flags:", overtakes[["teammates", "defender_slow", "wet_race"]].sum().to_dict())

print("\nRIVALS AT STRATEGY STOPS")
strategy = pit_stops[pit_stops["strategy_stop"]]
print(strategy["rival_side"].value_counts().reindex(sides).fillna(0).astype(int).to_string())

print("\nmissing values in laps (new columns):")
new_lap_columns = position_columns + ["red_flag_restart"]
print(laps[new_lap_columns].isna().sum()[lambda counts: counts > 0].to_string())
print("missing values in overtakes:")
print(overtakes.isna().sum()[lambda counts: counts > 0].to_string())

print("\nEXAMPLE ROWS: laps (new columns)")
print(laps.sample(3, random_state=42)[LAP_KEY + new_lap_columns].T.to_string())
print("\nEXAMPLE ROWS: overtakes")
print(overtakes.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: stops (new columns)")
print(pit_stops.sample(3, random_state=42)[KEY + ["in_lap", "strategy_stop", "position_before", "car_ahead", "car_behind",
                                                   "gap_ahead", "gap_behind", "ahead_is_teammate", "behind_is_teammate",
                                                   "rival_close", "rival_side"]].T.to_string())

print(f"\nRunning time: {time.time() - start_time:.1f} seconds")

CLEANING TABLE OF STAGE 2 (results/cleaning_table.csv)
                                                                       step  laps_before  laps_affected                                 action                                                                                                                                                                                                    reason
                                     combine the raw lap files of all races       101988              0                                   none                                                                                                                                                                                            starting point
                                                       exact duplicate rows       101988              0                                removed                                                                                                                 